In [ ]:
# Generate 2D random matrices from Gaussian random fields, then split them into 1D samples (sample groups)
'''
modified from code of Bruno Sciolla, https://github.com/bsciolla/gaussian-random-fields
'''

# Main dependencies
import numpy as np
import scipy.fftpack
import matplotlib
import matplotlib.pyplot as plt
import scipy.io as sio
import pandas as pd


def fftind(size):
    """ Returns a numpy array of shifted Fourier coordinates k_x k_y.
        
        Input args:
            size (integer): The size of the coordinate array to create
        Returns:
            k_ind, numpy array of shape (2, size, size) with:
                k_ind[0,:,:]:  k_x components
                k_ind[1,:,:]:  k_y components
                
        Example:
        
            print(fftind(5))
            
            [[[ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]]

            [[ 0  0  0  0  0]
            [ 1  1  1  1  1]
            [-3 -3 -3 -3 -3]
            [-2 -2 -2 -2 -2]
            [-1 -1 -1 -1 -1]]]
            
        """
    k_ind = np.mgrid[:size, :size] - int( (size + 1)/2 )
    k_ind = scipy.fftpack.fftshift(k_ind)
    return( k_ind )



# return numpy.ndarray of shape (size, size),
def gaussian_random_field(alpha = 6.5,# smooth factor
                          size = 64, # size of the field
                          mode = 'bound',# 'random' or bound
                          set_1 = 1.0, # if 'random', mean; else if 'bound', lower bound
                          set_2 = 1000.0):# if 'random', standard derivation; else if 'bound', upper bound
    """ Returns a numpy array of shifted Fourier coordinates k_x k_y.
        
        Input args:
            alpha (double, default = 3.0): 
                The power of the power-law momentum distribution
            size (integer, default = 128):
                The size of the square output Gaussian Random Fields
            flag_normalize (boolean, default = True):
                Normalizes the Gaussian Field:
                    - to have an average of 0.0
                    - to have a standard deviation of 1.0

        Returns:
            gfield (numpy array of shape (size, size)):
                The random gaussian random field
                
        Example:
        import matplotlib
        import matplotlib.pyplot as plt
        example = gaussian_random_field()
        plt.imshow(example)
        """
        
        # Defines momentum indices
    k_idx = fftind(size)

        # Defines the amplitude as a power law 1/|k|^(alpha/2)
    amplitude = np.power( k_idx[0]**2 + k_idx[1]**2 + 1e-10, -alpha/4.0 )
    amplitude[0,0] = 0
    
        # Draws a complex gaussian random noise with normal
        # (circular) distribution
    noise = np.random.normal(size = (size, size)) \
        + 1j * np.random.normal(size = (size, size))
    
        # To real space
    gfield = np.fft.ifft2(noise * amplitude).real
    
        # Sets the standard deviation to one
    gfield = gfield - np.mean(gfield)
    gfield = gfield/np.std(gfield)
    if mode == 'random':
        set_mean = set_1
        set_std  = set_2
        gfield = gfield * set_std
        gfield = gfield + set_mean
    elif mode == 'bound':
        set_lower = set_1
        set_upper = set_2
        g_max = np.max(gfield)
        g_min = np.min(gfield)
        gfield = (set_upper-set_lower)/(g_max - g_min) * gfield + \
                (set_lower*g_max - set_upper*g_min)/(g_max - g_min)
    else:
          raise KeyError("mode must be 'random' or 'bound")   
    return gfield




# def main():
#     import matplotlib
#     import matplotlib.pyplot as plt
#     example = gaussian_random_field()
#     plt.imshow(example, cmap='jet')
#     plt.savefig('example.png')
#     plt.show()

# if __name__ == '__main__':
#     main()

    # ------------ Core modification section ------------
    # rows, cols = 4, 4  # Subplot layout: 3 rows by 3 columns 1
    
    # Create the figure canvas and set an appropriate size to avoid crowded subplots
    # fig, axes = plt.subplots(rows, cols, figsize=(15, 12)) 1
    # fig.suptitle('9 Gaussian Random Fields (3×3 Layout)', fontsize=18, y=0.95) 1

num = 10000  # Generate 10,000 samples and pair them; the matching algorithm can be optimized and compared with or without duplicates
grf_array = np.zeros((64, 64, num))          # 1. Initialize an empty n-by-n-by-9 array
    
    
    # Generate random fields in a loop and plot the subplots
for i in range(num):
        # Generate the i-th random field
    grf = gaussian_random_field()
    grf_array[:, :, i] = grf                 # 2. Store the i-th field in the array during the loop
        
        # Compute the row and column indices of the subplot
        # row_idx = i // cols 1
        # col_idx = i % cols 1
        # Get the axis object for the current subplot
        # ax = axes[row_idx, col_idx] 1
        
        # Plot the subplot
        # im = ax.imshow(grf, cmap='jet') 1
        # Set the subplot title
        # ax.set_title(f'Field {i+1}', fontsize=12) 1
        # Hide axis ticks for a cleaner plot (optional)
        # ax.set_xticks([]) 1
        # ax.set_yticks([]) 1
        
        # Add a colorbar to each subplot; comment this out and use a global colorbar if using a shared color range
        # cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)  1
        # cbar.ax.tick_params(labelsize=8)  1
    
sio.savemat('grf_15.mat', {'grf_data': grf_array})  # 3. Save as a .mat file
   
    # Adjust subplot spacing to avoid overlap
    # plt.tight_layout(rect=[0, 0, 1, 0.92])  # The rect parameter reserves space for the title 1
    
    # Save the full figure to the current directory
    # plt.savefig('9_gaussian_random_fields.png', dpi=300, bbox_inches='tight') 1
    # Display the figure
    # plt.show() 1
X = grf_array.reshape(64*64, -1)   # (4096, 10000)

corr = np.corrcoef(X.T)            # (10000, 10000)


num = grf_array.shape[2]

threshold = 0.8
ssim_mat = np.abs(corr.copy())  # Copy the matrix, similar to the MATLAB code

dataX = []
dataY = []
S = []

for i in range(num):
    # Find sample indices whose correlation coefficient with sample i is greater than the threshold
    temp = np.where(ssim_mat[i, :] > threshold)[0]

    if len(temp) > 1:
        # Save the corresponding data
        dataX.append(grf_array[:, :, i])
        dataY.append(grf_array[:, :, temp[1]])  # MATLAB temp(2) -> Python temp[1]

        # Save the indices
        S.append([i, temp[1]])

        # Zero out the row corresponding to the matched sample to avoid duplicate matches
        ssim_mat[temp[1], :] = 0

# Convert to a NumPy array (optional)
S = np.array(S)

mask = (S[:, 0] != 0) & (S[:, 0] != S[:, 1])
S = S[mask]

model = np.zeros((S.shape[0]*64, 2, 64))


for i in range(S.shape[0]):
    for m in range(64):
        # x
        model[i*64 + m, 0, :] = grf_array[:, m, S[i][0]]
        # y
        model[i*64 + m, 1, :] = grf_array[:, m, S[i][1]]

np.save('model_data.npy', model)  # The saved samples are 10,000 pairs of 64-by-64 arrays

print(model.shape[0])
print(S)
pd.DataFrame(S).to_excel("np_array.xlsx", index=False, engine="openpyxl")

print(S.shape)

In [ ]:
# Forward modeling program
from MT1D_Forward import *
from joblib import Parallel, delayed  # Import the joblib parallel computing module
import numpy as np
from scipy.io import savemat

model_data = np.load('model_data.npy')  # The sample count changes each time the program above is run 

freqs = np.logspace(-2, 4, 64)    # Frequencies from 0.01 to 1000; this value controls the size


deepth = np.zeros((64, 1))  # Cumulative depth with 64 values in total

for i in range(63):   # i = 0 to 62, corresponding to MATLAB 1 to 63; the depth is 1950, and 63 is the number of layer thicknesses
    deepth[i+1, 0] = 20 + 10**(0.026026 * i) + deepth[i, 0]
                               
deepth = deepth.reshape(1, -1) 

resp = Parallel(n_jobs=-1, verbose=1)(       # This sets the number of cores; use -1 for the maximum, 1 to show the progress bar, and 0 to hide it
    delayed(Frequency_Domain_MT_Modeling)(4.0, frequencies=freqs, depth_arr=deepth, true_model=model_data[i]) 
    for i in range(624256)   # Number of 1D samples; 64000 / 64 gives the actual number of sample groups
    # for i in range(model_data.shape[0])

)


resp = np.array(resp)  # (n_samples, 4, n_freq), forward modeling results

print(model_data.shape)  # Total number of 1D samples; divide by 64 to get the number of sample groups

print(resp.shape)        # Number of forward responses

np.save('X_all.npy', resp)

Y_all = model_data[:624256]

np.save('Y_all.npy', Y_all)



In [6]:
# Verify sample shapes
import numpy as np

X_all = np.load('X_all.npy')
Y_all = np.load('Y_all.npy')
print(X_all.shape)
print(Y_all.shape)

(512000, 4, 64)
(512000, 2, 64)
